# Hierarchical Planning

## Imports

In [ ]:
import os
import time
import torch
import random
import minari
import numpy as np
import pprint

from dataclasses import dataclass
from torch import nn, Tensor
from torch.utils.data import DataLoader

# flow_matching
from flow_matching.path.scheduler import CondOTScheduler
from flow_matching.path import AffineProbPath
from flow_matching.solver import ODESolver
from flow_matching.utils import ModelWrapper

# visualization
import matplotlib.pyplot as plt
from matplotlib import cm

# training and evaluation
from src.run import (
    train
)

from src.pipelines.preprocessing import (
    collate_fn,
    get_dataset_stats,
    create_trajectory_chunks,
    create_normalized_chunks,
)
from src.pipelines.eval import (
    WrappedModel,
    WrappedConditionalModel,
    evaluate_open_loop,
    evaluate_policy_mpc,
)
from src.pipelines.sampling import generate_trajectory, unnormalize_trajectory
from src.models.backbone import MLP, CNN, ConditionalCNN, ConditionalUNet1D
from src.utils.loggers import WandBLogger

# visualization and evaluation
from src.pipelines.lunarlander.visualizers import visualize_trajectories as hvt
from src.pipelines.hopper.visualizers import visualize_trajectory as lvt

# To avoid meshgrid warning
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="torch")

if torch.cuda.is_available():
    device = "cuda:0"
    print("Using gpu")
else:
    device = "cpu"
    print("Using cpu.")
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

## Global Planner 

In [ ]:
from src.pipelines.lunarlander.visualizers import visualize_dataset
from src.conf.environment import LunarLanderConfig

config = LunarLanderConfig()
minari_dataset = minari.load_dataset(dataset_id=config.dataset_name)
minari_dataset_stats = get_dataset_stats(minari_dataset)
minari_dataset_stats

### CCNN

In [ ]:
@dataclass
class LunarLanderArgs:
    environment: str = "LunarLander-v3"
    horizon: int = 100
    batch_size: int = 32
    num_epochs: int = 100
    print_every: int = 10
    lr: float = 1e-3
    hidden_dim: int = 128
    kernel_size: int = 5
    step_size: float = 0.05
    solver_method: str = "midpoint"
    inference_batch_size: int = 1
    condition_on: str = "start_obs_goal"
    model_type: str = "ccnn"
    device: str = "cuda:0" if torch.cuda.is_available() else "cpu"
    chunk_type: str = "obs_only"
args = LunarLanderArgs()

# Preparing the dataset
minari_dataset = minari.load_dataset(dataset_id=config.dataset_name)
action_dim = config.action_dim
obs_dim = config.obs_dim
input_dim = obs_dim * args.horizon
cond_dim = obs_dim * 2
minari_dataset_stats = get_dataset_stats(minari_dataset)
dataloader = DataLoader(minari_dataset, batch_size=args.batch_size, shuffle=True, collate_fn=collate_fn)

SAVE_DIR = "src/checkpoints"
run_name = f"{args.environment}_{args.model_type}_h{args.horizon}_e{args.num_epochs}_k{args.kernel_size}_{args.condition_on}"
MODEL_NAME = run_name + ".pth"
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, MODEL_NAME)
pp = pprint.PrettyPrinter(indent=2)
print("Training configuration:")
pp.pprint(config)
pp.pprint(args)
print("Run name:", run_name)

logger = WandBLogger(
    config=args,
    run_name=run_name
)
model, stats, input_dim = train( 
    config=config, args=args, dataset=minari_dataset, logger=logger
)
env = minari_dataset.recover_environment()
evaluate_open_loop(env, model, stats, input_dim, args, logger=logger)
logger.finish()

### UNet

In [ ]:
@dataclass
class LunarLanderArgs:
    environment: str = "LunarLander-v3"
    horizon: int = 100
    batch_size: int = 32
    num_epochs: int = 100
    print_every: int = 10
    lr: float = 1e-3
    hidden_dim: int = 64
    kernel_size: int = 5
    step_size: float = 0.05
    solver_method: str = "midpoint"
    inference_batch_size: int = 1
    condition_on: str = "start_obs_goal"
    model_type: str = "unet"
    device: str = "cuda:0" if torch.cuda.is_available() else "cpu"
    chunk_type: str = "obs_only"
args = LunarLanderArgs()

# Preparing the dataset
minari_dataset = minari.load_dataset(dataset_id=config.dataset_name)
action_dim = config.action_dim
obs_dim = config.obs_dim
input_dim = obs_dim * args.horizon
cond_dim = obs_dim * 2
minari_dataset_stats = get_dataset_stats(minari_dataset)
dataloader = DataLoader(minari_dataset, batch_size=args.batch_size, shuffle=True, collate_fn=collate_fn)

SAVE_DIR = "src/checkpoints"
run_name = f"{args.environment}_{args.model_type}_h{args.horizon}_e{args.num_epochs}_k{args.kernel_size}_{args.condition_on}_GLOBAL"
MODEL_NAME = run_name + ".pth"
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, MODEL_NAME)
pp = pprint.PrettyPrinter(indent=2)
print("Training configuration:")
pp.pprint(config)
pp.pprint(args)
print("Run name:", run_name)

logger = WandBLogger(
    config=args,
    run_name=run_name
)
model, stats, input_dim = train( 
    config=config, args=args, dataset=minari_dataset, logger=logger
)
env = minari_dataset.recover_environment()
evaluate_open_loop(env, model, stats, input_dim, args, logger=logger)
logger.finish()

## Local Planner
This one predicts actions given observations generated by the global planner.

In [ ]:
@dataclass
class LunarLanderArgs:
    environment: str = "LunarLander-v3"
    horizon: int = 25
    batch_size: int = 32
    num_epochs: int = 100
    print_every: int = 10
    lr: float = 1e-3
    hidden_dim: int = 64
    kernel_size: int = 5
    step_size: float = 0.05
    solver_method: str = "midpoint"
    inference_batch_size: int = 1
    condition_on: str = "start_obs_goal"
    model_type: str = "unet"
    device: str = "cuda:0" if torch.cuda.is_available() else "cpu"
    chunk_type: str = "act_only"
args = LunarLanderArgs()

# Preparing the dataset
minari_dataset = minari.load_dataset(dataset_id=config.dataset_name)
action_dim = config.action_dim
obs_dim = config.obs_dim
input_dim = action_dim * args.horizon
cond_dim = obs_dim * 2
minari_dataset_stats = get_dataset_stats(minari_dataset)
dataloader = DataLoader(minari_dataset, batch_size=args.batch_size, shuffle=True, collate_fn=collate_fn)

SAVE_DIR = "src/checkpoints"
run_name = f"{args.environment}_{args.model_type}_h{args.horizon}_e{args.num_epochs}_k{args.kernel_size}_{args.condition_on}_LOCAL"
MODEL_NAME = run_name + ".pth"
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, MODEL_NAME)
pp = pprint.PrettyPrinter(indent=2)
print("Training configuration:")
pp.pprint(config)
pp.pprint(args)
print("Run name:", run_name)

In [ ]:
logger = WandBLogger(
    config=args,
    run_name=run_name
)
model, stats, input_dim = train( 
    config=config, args=args, dataset=minari_dataset, logger=logger
)
logger.finish()

## MPC Evaluation
1. call the global planner to get a sequence of observations
2. use the output of the global planner as conditioning signals for the local planner

In [16]:
MODEL_PATH = "src/checkpoints/lunarlander_unet_h10_e100_k5_start_obs_goal_GLOBAL.pth"